[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Time-Series-Analysis/blob/main/Quantlets/Ch_07/TSA_ch7_engle_granger/TSA_ch7_engle_granger.ipynb)

# TSA_ch7_engle_granger

The Engle-Granger two-step method: the null distribution of the residual-based Dickey-Fuller statistic for 2 and 3 variables by simulation, against the Dickey-Fuller distribution and the MacKinnon (2010) critical values, with the size of the test when the Dickey-Fuller value is used; Engle-Granger and Phillips-Ouliaris tests on US consumption and income, US yields, two Romanian banks, EUR/RON and EUR/HUF and Romanian consumption and GDP; the step-1 residuals and their ACF.

*Time Series Analysis (TSA), Chapter 7: Cointegration and VECM. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/TSA/Chapter_7'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# The arch package (Phillips-Ouliaris test) is not preinstalled in Google Colab: pip install arch
import importlib.util, subprocess, sys
if importlib.util.find_spec('arch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'arch'])

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import tsa_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Time-Series-Analysis/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the TSA repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
import itertools

import statsmodels.api as sm

from statsmodels.tsa.stattools import adfuller, coint, acf

from statsmodels.tsa.adfvalues import mackinnoncrit, mackinnonp

from statsmodels.tsa.vector_ar.vecm import VECM, coint_johansen, select_order

from statsmodels.tsa.api import VAR

from arch.unitroot.cointegration import phillips_ouliaris

SEED = 2026

YIELDS = ['GS1', 'GS5', 'GS10']

YIELD_START = '1960-01-01'

HICP_RO = ('prc_hicp_minr', 'M.I15.TOTAL.RO')

HICP_EA = ('prc_hicp_minr', 'M.I15.TOTAL.EA')

ROBOR = ('irt_st_m', 'M.IRT_M3.RO')

EURIBOR = ('irt_st_m', 'M.IRT_M3.EA')

CONS_RO = ('namq_10_gdp', 'Q.CLV10_MEUR.SCA.P31_S14.RO')

GDP_RO = ('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')

BVB = ['TLV.RO', 'BRD.RO', 'SNP.RO', 'TGN.RO', 'TEL.RO', 'EL.RO', 'SNG.RO', 'SNN.RO']

US_BANKS = ['JPM.US', 'BAC.US', 'C.US', 'WFC.US', 'GS.US', 'MS.US']

PAIR = ['TLV.RO', 'BRD.RO']

PAIR_START = '2014-01-01'

FORM, TRADE, ENTRY = 252, 126, 2.0

COST_BVB, COST_US = 0.002, 0.0005

def us_yields(cols=YIELDS, start=YIELD_START):
    """US Treasury constant-maturity yields (FRED, monthly averages, % per year), monthly PeriodIndex."""
    f = read_fred(cols).dropna().loc[start:]
    f.index = pd.PeriodIndex(f.index, freq='M')
    return f


def us_macro():
    """US quarterly real GDP, consumption and investment (statsmodels macrodata, 1959Q1-2009Q3), 100 ln."""
    d = load_statsmodels('macrodata')
    return 100 * np.log(d[['realgdp', 'realcons', 'realinv']]).rename(
        columns={'realgdp': 'y', 'realcons': 'c', 'realinv': 'i'})


def cee_fx(start='2005-07-01'):
    """EUR/RON, EUR/HUF and EUR/PLN (month-end, from the BNR reference rates: EUR/HUF = EUR/RON / HUF/RON), 100 ln."""
    e, h, p = (read_reference_rate(c, start=start) for c in ('EUR', 'HUF', 'PLN'))
    d = pd.concat([e, h, p], axis=1, keys=['eur', 'huf', 'pln']).dropna()
    fx = pd.DataFrame({'EUR/RON': d.eur, 'EUR/HUF': 100 * d.eur / d.huf, 'EUR/PLN': d.eur / d.pln})
    return 100 * np.log(fx.resample('ME').last())


def stock(symbol, start):
    """Adjusted close of a stock from data/market (weekdays)."""
    t = read_market(symbol)
    s = pd.to_numeric(t['adjusted_close'], errors='coerce').loc[start:'2026-09-18'].dropna()
    s = s[s > 0]
    return s[s.index.dayofweek < 5].rename(symbol.split('.')[0])


def stock_panel(symbols, start, calendar='bet'):
    """Several stocks on a common calendar (BET or S&P 500 trading days); a missing quote is carried forward
    for at most 5 days (no trade that day), then the rows with any missing value are dropped."""
    cal = load_close(calendar, start=start).index
    P = pd.concat([stock(s, start) for s in symbols], axis=1).reindex(cal).ffill(limit=5)
    return P.dropna()


def save(name, save_it=True):
    if save_it:
        st.check_no_grey(plt.gcf())
        st.save_fig(name)
    else:
        plt.show()


def years_axis(ax, step=5):
    import matplotlib.dates as mdates
    ax.xaxis.set_major_locator(mdates.YearLocator(step))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))


def ts_index(x):
    """A DatetimeIndex for plotting (PeriodIndex -> timestamps)."""
    return x.index.to_timestamp() if isinstance(x.index, pd.PeriodIndex) else x.index


def adf_test(x, reg='c'):
    """ADF test (lags by AIC): statistic, MacKinnon p-value, lags, 5% critical value."""
    r = adfuller(np.asarray(pd.Series(x).dropna(), float), regression=reg, autolag='AIC')
    return {'stat': float(r[0]), 'p': float(r[1]), 'lags': int(r[2]), 'crit5': float(r[4]['5%'])}


def ols(y, X):
    """OLS with a constant: coefficients, standard errors, t-statistics, residuals, R^2, DW."""
    m = sm.OLS(np.asarray(y, float), sm.add_constant(np.asarray(X, float))).fit()
    e = m.resid
    return {'b': m.params.tolist(), 'se': m.bse.tolist(), 't': m.tvalues.tolist(), 'r2': float(m.rsquared),
            'dw': float(np.sum(np.diff(e) ** 2) / np.sum(e ** 2)), 'resid': e, 'n': int(m.nobs)}


def eg_test(y, x, trend='c'):
    """Engle-Granger two-step test: OLS of y on a constant and x (step 1), ADF on the residuals with lags by AIC
    and MacKinnon (2010) p-values for len(x)+1 variables (step 2); Phillips-Ouliaris Z_t on the same regression."""
    y = pd.Series(y)
    X = pd.DataFrame(x)
    d = pd.concat([y, X], axis=1).dropna()
    yv, Xv = d.iloc[:, 0], d.iloc[:, 1:]
    stat, p, cv = coint(yv, Xv, trend=trend, autolag='aic')
    step1 = ols(yv, Xv)
    po = phillips_ouliaris(yv, Xv, trend=trend, test_type='Zt')
    lags = adfuller(step1['resid'], regression='n', autolag='AIC')[2]
    return {'n': int(len(d)), 'beta': [float(b) for b in step1['b'][1:]], 'const': float(step1['b'][0]),
            'r2': step1['r2'], 'dw': step1['dw'], 'tau': float(stat), 'p': float(p), 'crit5': float(cv[1]),
            'crit1': float(cv[0]), 'crit10': float(cv[2]), 'lags': int(lags), 'po_zt': float(po.stat),
            'po_p': float(po.pvalue), 'po_crit5': float(po.critical_values[5]),
            'adf_crit5': float(mackinnoncrit(N=1, regression='c', nobs=len(d))[1])}


def half_life(rho):
    """Half-life (periods) of a deviation that decays like rho^h: ln(0.5)/ln(rho), for 0 < rho < 1."""
    return float(np.log(0.5) / np.log(rho)) if 0 < rho < 1 else float('inf')


def johansen(df, det_order=0, k_ar_diff=None, maxlags=8):
    """Johansen trace and maximum-eigenvalue tests (MacKinnon-Haug-Michelis critical values via statsmodels).
    k_ar_diff = lags of the differences in the VECM, by BIC on the levels VAR if None. The rank is the first r whose
    null is not rejected at 5% in the sequence r = 0, 1, ..."""
    x = df.dropna()
    if k_ar_diff is None:
        k_ar_diff = int(max(select_order(x, maxlags=maxlags, deterministic='co').bic, 1))
    r = coint_johansen(x, det_order, k_ar_diff)
    k = x.shape[1]
    rank = lambda s, c: next((i for i in range(k) if s[i] < c[i, 1]), k)
    return {'n': int(len(x)), 'k_ar_diff': k_ar_diff, 'eig': r.eig.tolist(), 'trace': r.lr1.tolist(),
            'trace_cv5': r.cvt[:, 1].tolist(), 'maxeig': r.lr2.tolist(), 'maxeig_cv5': r.cvm[:, 1].tolist(),
            'rank_trace': rank(r.lr1, r.cvt), 'rank_maxeig': rank(r.lr2, r.cvm), 'first': str(x.index[0])[:10],
            'last': str(x.index[-1])[:10]}


def ecm_fit(y, x, lags=1):
    """Single-equation ECM (two steps): e_t = y_t - a - b x_t from the levels regression, then
    Delta y_t = c + gamma e_{t-1} + delta_0 Delta x_t + sum_j (phi_j Delta y_{t-j} + delta_j Delta x_{t-j}) + eps_t."""
    d = pd.concat([pd.Series(y, name='y'), pd.Series(x, name='x')], axis=1).dropna()
    s1 = ols(d.y, d.x)
    e = pd.Series(s1['resid'], index=d.index)
    Z = pd.DataFrame({'e1': e.shift(1), 'dx': d.x.diff()})
    for j in range(1, lags + 1):
        Z[f'dy{j}'] = d.y.diff().shift(j)
        Z[f'dx{j}'] = d.x.diff().shift(j)
    Z['dy'] = d.y.diff()
    Z = Z.dropna()
    m = sm.OLS(Z.dy, sm.add_constant(Z.drop(columns='dy'))).fit()
    g = float(m.params['e1'])
    return {'beta': float(s1['b'][1]), 'a': float(s1['b'][0]), 'gamma': g, 'gamma_se': float(m.bse['e1']),
            'gamma_t': float(m.tvalues['e1']), 'delta0': float(m.params['dx']), 'delta0_se': float(m.bse['dx']),
            'const': float(m.params['const']), 'r2': float(m.rsquared), 'n': int(m.nobs), 'half': half_life(1 + g),
            'params': {k: float(v) for k, v in m.params.items()}, 'tvalues': {k: float(v) for k, v in m.tvalues.items()}}


def eg_tau_batch(n_var, T, R, rng):
    """R values of the Dickey-Fuller t-statistic (no lags, no constant in the test regression) on the residuals of
    the OLS regression of one random walk on n_var - 1 other, independent random walks and a constant; n_var = 1:
    the Dickey-Fuller test with a constant on a single random walk."""
    out = np.empty(R)
    for r in range(R):
        W = rng.standard_normal((T, n_var)).cumsum(axis=0)
        if n_var == 1:
            e = W[:, 0] - W[:, 0].mean()
        else:
            X = np.column_stack([np.ones(T), W[:, 1:]])
            e = W[:, 0] - X @ np.linalg.lstsq(X, W[:, 0], rcond=None)[0]
        de, el = np.diff(e), e[:-1]
        g = (el @ de) / (el @ el)
        s2 = np.sum((de - g * el) ** 2) / (len(de) - 1)
        out[r] = g / np.sqrt(s2 / (el @ el))
    return out


def fig_eg_dist(T=200, R=10000, save_it=True):
    """Null distributions (no cointegration) of the residual-based Dickey-Fuller statistic for 2 and 3 variables,
    against the Dickey-Fuller distribution with a constant (one variable) and the N(0, 1); empirical 5% quantiles,
    MacKinnon critical values and the rejection rate of the Engle-Granger test when the Dickey-Fuller value is used."""
    rng = np.random.default_rng(SEED)
    taus = {k: eg_tau_batch(k, T, R, rng) for k in (1, 2, 3)}
    fig, ax = plt.subplots(figsize=(9.6, 3.6))
    grid = np.linspace(-6, 3, 400)
    from scipy.stats import gaussian_kde, norm
    ax.plot(grid, norm.pdf(grid), color=st.DarkText, lw=1.0, ls='--', label='N(0, 1)')
    cols = {1: st.MainBlue, 2: st.IDAred, 3: st.Forest}
    labs = {1: 'Dickey-Fuller, one series (constant)', 2: 'Engle-Granger, 2 variables', 3: 'Engle-Granger, 3 variables'}
    out = {'T': T, 'R': R}
    for k in (1, 2, 3):
        ax.plot(grid, gaussian_kde(taus[k])(grid), color=cols[k], lw=1.4, label=labs[k])
        q = float(np.quantile(taus[k], 0.05))
        ax.axvline(q, color=cols[k], lw=0.9, ls=':')
        out[k] = {'q05': q, 'q01': float(np.quantile(taus[k], 0.01)), 'q10': float(np.quantile(taus[k], 0.10)),
                  'mk5': float(mackinnoncrit(N=k, regression='c', nobs=T)[1]),
                  'mk1': float(mackinnoncrit(N=k, regression='c', nobs=T)[0]),
                  'mk10': float(mackinnoncrit(N=k, regression='c', nobs=T)[2])}
    for k in (2, 3):
        out[k]['size_with_df'] = float(np.mean(taus[k] < out[1]['mk5']))
    ax.set_xlim(-6, 3)
    ax.set_title(f'Null distributions of the t-statistic (T = {T}, {R:,} simulations); dotted: 5% quantiles')
    ax.set_xlabel('t-statistic')
    st.legend_outside_bottom(ax, ncol=2, y=-0.2)
    plt.tight_layout()
    save('tsa_ch7_eg_dist', save_it)
    # asymptotic table of MacKinnon (2010) critical values with a constant
    out['table'] = {k: [float(v) for v in mackinnoncrit(N=k, regression='c', nobs=np.inf)] for k in (1, 2, 3, 4)}
    out['table_ct'] = {k: [float(v) for v in mackinnoncrit(N=k, regression='ct', nobs=np.inf)] for k in (1, 2, 3, 4)}
    return out


def eg_table(save_it=True):
    """Engle-Granger and Phillips-Ouliaris tests on six pairs; the table goes to ch7_eg_tests.csv."""
    f = read_fred(['TB3MS', 'GS10']).dropna().loc[YIELD_START:]
    m = us_macro()
    fx = cee_fx()
    P = 100 * np.log(stock_panel(PAIR, PAIR_START))
    ro = 100 * np.log(pd.concat([read_eurostat(*CONS_RO), read_eurostat(*GDP_RO)], axis=1, keys=['c', 'y']).dropna())
    cases = [('US consumption on income', m.c, m.y), ('US 10-year on 3-month yield', f.GS10, f.TB3MS),
             ('TLV on BRD', P.TLV, P.BRD),
             ('EUR/RON on EUR/HUF', fx['EUR/RON'], fx['EUR/HUF']), ('Romania consumption on GDP', ro.c, ro.y)]
    out = {name: eg_test(a, b) for name, a, b in cases}
    out['TLV on BRD']['first'] = str(P.index[0])[:10]
    P10 = 100 * np.log(stock_panel(PAIR, '2010-01-01'))
    out['TLV on BRD, from 2010'] = eg_test(P10.TLV, P10.BRD)
    out['TLV on BRD, reversed'] = eg_test(P.BRD, P.TLV)
    out['US income on consumption'] = eg_test(m.y, m.c)
    ro_first = str(ro.index[0])[:10]
    if save_it:
        pd.DataFrame({k: {'T': v['n'], 'beta': v['beta'][0], 'EG tau': v['tau'], 'EG p': v['p'], 'EG 5%': v['crit5'],
                          'PO Zt': v['po_zt'], 'PO p': v['po_p']} for k, v in out.items()}).T.round(4).to_csv(
            os.path.join(globals().get('HERE', '.'), 'ch7_eg_tests.csv'))
    out['ro_first'] = ro_first
    return out


def fig_eg_steps(save_it=True):
    """Step 1 residuals of two Engle-Granger regressions: US consumption on income (quarterly) and the 10-year on the
    3-month yield (monthly); their sample ACF against that of a random walk."""
    m = us_macro()
    m.index = pd.period_range('1959Q1', periods=len(m), freq='Q').to_timestamp()
    f = read_fred(['TB3MS', 'GS10']).dropna().loc[YIELD_START:]
    e1 = pd.Series(ols(m.c, m.y)['resid'], index=m.index)
    e2 = pd.Series(ols(f.GS10, f.TB3MS)['resid'], index=f.index)
    fig, ax = plt.subplots(1, 3, figsize=(10.6, 3.2), gridspec_kw={'width_ratios': [1.2, 1.2, 1]})
    ax[0].plot(e1.index, e1, color=st.Purple, lw=0.9, label='_nolegend_')
    ax[0].axhline(0, color=st.DarkText, lw=0.6)
    ax[0].set_title('Consumption on income: residuals')
    years_axis(ax[0], 10)
    ax[1].plot(e2.index, e2, color=st.IDAred, lw=0.8, label='_nolegend_')
    ax[1].axhline(0, color=st.DarkText, lw=0.6)
    ax[1].set_title('10-year on 3-month yield: residuals')
    years_axis(ax[1], 10)
    lags = np.arange(1, 25)
    rw = np.random.default_rng(SEED).standard_normal(len(e2)).cumsum()
    ax[2].plot(lags, acf(e1, nlags=24)[1:], color=st.Purple, marker='o', ms=3, lw=1, label='consumption residuals (quarters)')
    ax[2].plot(lags, acf(e2, nlags=24)[1:], color=st.IDAred, marker='s', ms=3, lw=1, label='yield residuals (months)')
    ax[2].plot(lags, acf(rw, nlags=24)[1:], color=st.MainBlue, lw=1, ls='--', label='a random walk')
    ax[2].axhline(0, color=st.DarkText, lw=0.6)
    ax[2].set_title('Sample ACF')
    ax[2].set_xlabel('lag')
    st.fig_legend_bottom(fig, ncol=3, y=-0.01)
    plt.tight_layout()
    save('tsa_ch7_eg_steps', save_it)
    return {'acf1_cons': float(acf(e1, nlags=1)[1]), 'acf1_rates': float(acf(e2, nlags=1)[1]),
            'acf12_rates': float(acf(e2, nlags=12)[12]), 'acf8_cons': float(acf(e1, nlags=8)[8]),
            'sd_cons': float(e1.std()), 'sd_rates': float(e2.std())}

## Run

In [ ]:
print(fig_eg_dist())
E = eg_table()
print(pd.DataFrame({k: {"beta": v["beta"][0], "tau": v["tau"], "p": v["p"], "PO Zt": v["po_zt"], "PO p": v["po_p"]} for k, v in E.items() if isinstance(v, dict)}).T)
print(fig_eg_steps())

![tsa_ch7_eg_dist](./tsa_ch7_eg_dist.png)

Output: `tsa_ch7_eg_dist.pdf`

![tsa_ch7_eg_steps](./tsa_ch7_eg_steps.png)

Output: `tsa_ch7_eg_steps.pdf`